# Trabalho Prático 1 - INF01017
## Spot-checking de Algoritmos
### Dataset: Ames Mutagenicity

**Objetivo:** Avaliar rapidamente múltiplos algoritmos de ML para identificar os mais promissores.

---

## 1. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
import warnings

warnings.filterwarnings('ignore')

# Configurações
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 2. Import dos Algoritmos

In [ ]:
# Algoritmos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

## 3. Carregamento dos Dados Processados

In [ ]:
# Carregar dados de treino e teste
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

# Separar features e target
X_train = train_df.drop(columns=['Overall'])
y_train = train_df['Overall']
X_test = test_df.drop(columns=['Overall'])
y_test = test_df['Overall']

print(f"Shape X_train: {X_train.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_test: {y_test.shape}")

## 4. Definição dos Modelos

In [ ]:
# Dicionário com os modelos a serem testados
models = {
    # Modelos Lineares
    'Logistic Regression': LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    'SVM Linear': SVC(kernel='linear', random_state=RANDOM_SEED, probability=True),
    
    # Modelos Baseados em Árvores
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100),
    
    # Boosting
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_SEED, n_estimators=100),
    
    # Baseado em Instâncias
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    
    # Probabilístico
    'Naive Bayes': GaussianNB(),
    
    # Redes Neurais
    'MLP Neural Network': MLPClassifier(random_state=RANDOM_SEED, max_iter=500, 
                                        hidden_layer_sizes=(100, 50))
}

print(f"Número de modelos definidos: {len(models)}")
print("\nModelos:")
for i, name in enumerate(models.keys(), 1):
    print(f"  {i}. {name}")

## 5. Função de Avaliação com Cross-Validation

In [ ]:
def evaluate_model_cv(model, X, y, cv_folds=5):
    """
    Avalia modelo usando cross-validation.
    """
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_SEED)
    
    # Calcular métricas
    accuracy_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    precision_scores = cross_val_score(model, X, y, cv=cv, scoring='precision')
    recall_scores = cross_val_score(model, X, y, cv=cv, scoring='recall')
    f1_scores = cross_val_score(model, X, y, cv=cv, scoring='f1')
    roc_auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    
    results = {
        'accuracy_mean': accuracy_scores.mean(),
        'accuracy_std': accuracy_scores.std(),
        'precision_mean': precision_scores.mean(),
        'precision_std': precision_scores.std(),
        'recall_mean': recall_scores.mean(),
        'recall_std': recall_scores.std(),
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'roc_auc_mean': roc_auc_scores.mean(),
        'roc_auc_std': roc_auc_scores.std()
    }
    
    return results

## 6. Spot-checking dos Modelos

In [ ]:
# Realizar spot-checking
cv_folds = 5
results = []

print(f"Iniciando spot-checking com {cv_folds}-fold cross-validation...\n")

for name, model in models.items():
    print(f"Avaliando: {name}...", end=' ')
    try:
        metrics = evaluate_model_cv(model, X_train, y_train, cv_folds)
        metrics['model'] = name
        results.append(metrics)
        print(f"✓ Accuracy: {metrics['accuracy_mean']:.4f} (±{metrics['accuracy_std']:.4f})")
    except Exception as e:
        print(f"✗ ERRO: {str(e)}")

# Criar DataFrame com resultados
results_df = pd.DataFrame(results)

# Reordenar colunas
cols = ['model', 'accuracy_mean', 'accuracy_std', 'precision_mean', 'precision_std',
        'recall_mean', 'recall_std', 'f1_mean', 'f1_std', 'roc_auc_mean', 'roc_auc_std']
results_df = results_df[cols]

print("\n" + "="*60)
print("SPOT-CHECKING CONCLUÍDO!")
print("="*60)

## 7. Visualização dos Resultados

In [ ]:
# Tabela de resultados
print("\nResultados completos:")
results_df

In [ ]:
# Gráficos comparativos
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, metric in enumerate(metrics):
    mean_col = f'{metric}_mean'
    std_col = f'{metric}_std'
    
    results_sorted = results_df.sort_values(mean_col, ascending=False)
    
    axes[idx].barh(results_sorted['model'], results_sorted[mean_col], 
                  xerr=results_sorted[std_col], color='steelblue', alpha=0.7)
    axes[idx].set_xlabel('Score')
    axes[idx].set_title(f'{metric.upper()}', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='x')
    axes[idx].set_xlim(0, 1)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## 8. Ranking dos Modelos

In [ ]:
# Ranquear por F1-Score (métrica principal)
ranked_df = results_df.sort_values('f1_mean', ascending=False).reset_index(drop=True)
ranked_df['rank'] = ranked_df.index + 1

print("\n" + "="*60)
print("RANKING DOS MODELOS - Métrica: F1-SCORE")
print("="*60)

for idx, row in ranked_df.head(3).iterrows():
    print(f"\n{idx+1}º lugar: {row['model']}")
    print(f"   Accuracy: {row['accuracy_mean']:.4f} (±{row['accuracy_std']:.4f})")
    print(f"   Precision: {row['precision_mean']:.4f} (±{row['precision_std']:.4f})")
    print(f"   Recall: {row['recall_mean']:.4f} (±{row['recall_std']:.4f})")
    print(f"   F1-Score: {row['f1_mean']:.4f} (±{row['f1_std']:.4f})")
    print(f"   ROC-AUC: {row['roc_auc_mean']:.4f} (±{row['roc_auc_std']:.4f})")

## 9. Salvamento dos Resultados

In [ ]:
import os

# Salvar resultados em CSV
output_path = '../results/metrics/'
os.makedirs(output_path, exist_ok=True)

filepath = os.path.join(output_path, 'spot_checking_results.csv')
ranked_df.to_csv(filepath, index=False)

print(f"\nResultados salvos em: {filepath}")

## 10. Avaliação no Conjunto de Teste (Top 3 Modelos)

In [ ]:
# Treinar e avaliar os top 3 modelos no conjunto de teste
print("\n" + "="*60)
print("AVALIAÇÃO NO CONJUNTO DE TESTE - TOP 3 MODELOS")
print("="*60)

top_3_models = ranked_df.head(3)['model'].tolist()

for model_name in top_3_models:
    print(f"\n{'='*60}")
    print(f"Modelo: {model_name}")
    print(f"{'='*60}")
    
    # Treinar modelo
    model = models[model_name]
    model.fit(X_train, y_train)
    
    # Predições
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Métricas
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    if y_pred_proba is not None:
        print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
    
    # Relatório de classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))
    
    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusão - {model_name}')
    plt.ylabel('Verdadeiro')
    plt.xlabel('Predito')
    plt.show()

## 11. Conclusões

**Escreva aqui as conclusões do spot-checking:**

1. Quais foram os 2-3 algoritmos mais promissores?
2. Qual métrica foi mais relevante para a decisão?
3. Houve grande variação de desempenho entre os modelos?
4. Algum modelo teve desempenho muito superior aos demais?
5. Recomendações para o T2 (otimização de hiperparâmetros)

---

**Próximo passo:** Trabalho Prático 2 - Otimização de hiperparâmetros e análise aprofundada dos modelos selecionados.

---